In [2]:
import tkinter as tk
from tkinter import filedialog, messagebox
import threading
import sounddevice as sd
import soundfile as sf
import numpy as np
import librosa
import joblib
import os
import datetime
import csv

# -------------------------
# Configuration / Filenames
# -------------------------
MODEL_PATH = "voice_model.pkl"
SCALER_PATH = "voice_scaler.pkl"
ENCODER_PATH = "voice_encoder.pkl"
HISTORY_CSV = "history.csv"
TEMP_RECORDING = "temp_recording.wav"

# -------------------------
# Load model, scaler, encoder
# -------------------------
try:
    model = joblib.load(MODEL_PATH)
except Exception as e:
    model = None
    print(f"Warning: Could not load model from {MODEL_PATH}: {e}")

try:
    scaler = joblib.load(SCALER_PATH)
except Exception as e:
    scaler = None
    print(f"Warning: Could not load scaler from {SCALER_PATH}: {e}")

try:
    encoder = joblib.load(ENCODER_PATH)
except Exception as e:
    encoder = None
    print(f"Warning: Could not load encoder from {ENCODER_PATH}: {e}")

# -------------------------
# Utility functions
# -------------------------
def append_history(name, confidence, source):
    """Append a prediction record to CSV history."""
    header_needed = not os.path.exists(HISTORY_CSV)
    now = datetime.datetime.now().isoformat(sep=' ', timespec='seconds')
    with open(HISTORY_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if header_needed:
            writer.writerow(["timestamp", "source", "prediction", "confidence"])
        writer.writerow([now, source, name, f"{confidence:.4f}"])

def safe_load_audio(path, sr=22050):
    """Load audio with librosa; return mono waveform and sample rate."""
    audio, sample_rate = librosa.load(path, sr=sr, mono=True)
    return audio, sample_rate

def preprocess_audio(y, sr):
    """
    Basic preprocessing:
    - trim leading/trailing silence
    - apply a small pre-emphasis
    Returns processed audio.
    """
    # Trim silence (top_db can be adjusted)
    y_trimmed, _ = librosa.effects.trim(y, top_db=20)
    # Pre-emphasis
    pre_emph = 0.97
    y_emph = np.append(y_trimmed[0], y_trimmed[1:] - pre_emph * y_trimmed[:-1])
    return y_emph

def extract_mfcc(file_path, n_mfcc=20):
    """Return MFCC mean vector with shape (1, n_mfcc)."""
    y, sr = safe_load_audio(file_path, sr=22050)
    if y.size == 0:
        raise ValueError("Loaded audio is empty.")
    y = preprocess_audio(y, sr)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfccs_mean = np.mean(mfccs.T, axis=0)  # shape (n_mfcc,)
    return mfccs_mean.reshape(1, -1)       # shape (1, n_mfcc)

def predict_sound(path):
    """Run pipeline: extract MFCC -> scale -> predict -> return name and confidence."""
    if model is None or scaler is None or encoder is None:
        raise RuntimeError("Model, scaler or encoder not loaded. See console output for issues.")

    mfcc = extract_mfcc(path)           # shape (1, n_mfcc)
    # scaler expects 2D array with shape (n_samples, n_features)
    mfcc_scaled = scaler.transform(mfcc)  # shape (1, n_features)

    # Prediction
    try:
        # If classifier supports predict_proba
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(mfcc_scaled)[0]
            pred_idx = int(np.argmax(probs))
            confidence = float(np.max(probs))
        else:
            pred_idx = int(model.predict(mfcc_scaled)[0])
            confidence = 1.0  # no probability available
    except Exception as e:
        # If model outputs labels directly (strings), skip encoder inverse
        raise RuntimeError(f"Prediction failed: {e}")

    # Map back to label name using encoder
    try:
        name = encoder.inverse_transform([pred_idx])[0]
    except Exception:
        # If encoder is not a sklearn LabelEncoder but model already outputs string label
        try:
            name = model.predict(mfcc_scaled)[0]
            confidence = 1.0
        except Exception:
            raise RuntimeError("Could not map predicted index to label name. Check encoder type.")

    return name, confidence

# -------------------------
# Recorder Threading
# -------------------------
recording_thread = None
_is_recording = False

def _record_thread(duration_sec, filename):
    global _is_recording
    try:
        _is_recording = True
        btn_record.config(state="disabled")
        label_status.config(text="Recording...")
        root.update()

        fs = 22050
        # Record (blocking inside thread)
        audio = sd.rec(int(duration_sec * fs), samplerate=fs, channels=1, dtype='float32')
        sd.wait()
        # Save as WAV (float32 supported by soundfile)
        sf.write(filename, audio, fs, subtype='PCM_16')
        label_status.config(text=f"Recorded & Saved: {filename}")
        root.update()
        # After recording, predict automatically
        try:
            name, conf = predict_sound(filename)
            label_result.config(text=f"Prediction: {name} ({conf*100:.2f}%)")
            append_history(name, conf, "record")
        except Exception as e:
            messagebox.showerror("Prediction Error", str(e))
    except Exception as e:
        messagebox.showerror("Recording Error", str(e))
    finally:
        _is_recording = False
        btn_record.config(state="normal")

def start_record(duration=3):
    """Start recording in a background thread."""
    global recording_thread
    if _is_recording:
        messagebox.showinfo("Info", "Already recording.")
        return
    recording_thread = threading.Thread(target=_record_thread, args=(duration, TEMP_RECORDING), daemon=True)
    recording_thread.start()

def stop_recording():
    """Stop ongoing recording (best-effort)."""
    try:
        sd.stop()
        label_status.config(text="Recording stopped.")
    except Exception as e:
        messagebox.showerror("Stop Error", str(e))

# -------------------------
# Play / Stop audio
# -------------------------
_play_thread = None
_is_playing = False

def _play_thread(path):
    global _is_playing
    try:
        _is_playing = True
        data, fs = sf.read(path, dtype='float32')
        sd.play(data, fs)
        sd.wait()
    except Exception as e:
        messagebox.showerror("Playback Error", str(e))
    finally:
        _is_playing = False

def play_audio(path):
    global _play_thread
    if not os.path.exists(path):
        messagebox.showerror("Error", f"File not found: {path}")
        return
    if _is_playing:
        messagebox.showinfo("Info", "Audio already playing.")
        return
    _play_thread = threading.Thread(target=_play_thread, args=(path,), daemon=True)
    _play_thread.start()

def stop_audio():
    try:
        sd.stop()
        label_status.config(text="Playback stopped.")
    except Exception as e:
        messagebox.showerror("Stop Error", str(e))

# -------------------------
# UI Callback Handlers
# -------------------------
def on_record_button():
    # You can let user choose duration or keep default (3 sec)
    start_record(duration=3)

def on_upload_button():
    try:
        file_path = filedialog.askopenfilename(
            title="Select Audio File",
            filetypes=[("Audio Files", "*.wav *.mp3 *.flac *.ogg")]
        )
        if not file_path:
            return
        label_status.config(text=f"Uploaded: {os.path.basename(file_path)}")
        root.update()
        # Convert to consistent WAV (optional) and save a temp copy for playback/prediction
        # We will just use the original file for prediction (librosa handles mp3/flac)
        name, conf = predict_sound(file_path)
        label_result.config(text=f"Prediction: {name} ({conf*100:.2f}%)")
        append_history(name, conf, "upload")
    except Exception as e:
        messagebox.showerror("Error", str(e))

def on_play_last():
    if os.path.exists(TEMP_RECORDING):
        play_audio(TEMP_RECORDING)
    else:
        messagebox.showinfo("Info", "No recording found. Please record first.")

def show_history():
    if not os.path.exists(HISTORY_CSV):
        messagebox.showinfo("History", "No history found yet.")
        return
    # Open the CSV and build a simple string preview
    with open(HISTORY_CSV, "r", encoding="utf-8") as f:
        lines = f.read().strip().splitlines()
    # show last 10 lines
    preview = "\n".join(lines[-11:])  # header + last 10
    messagebox.showinfo("Prediction History (last 10)", preview)

# -------------------------
# Tkinter UI
# -------------------------
root = tk.Tk()
root.title("Voice Recognition App — Professional")
root.geometry("480x420")
root.resizable(False, False)

# Top label
header = tk.Label(root, text="✨ Namaste 🙏🌺", font=("Helvetica", 18, "bold"))
header.pack(pady=12)

# Buttons frame
frame_buttons = tk.Frame(root)
frame_buttons.pack(pady=6)

btn_record = tk.Button(frame_buttons, text="🎤 Record Voice (3 sec)", command=on_record_button,
                       font=("Arial", 12), width=22)
btn_record.grid(row=0, column=0, padx=6, pady=6)

btn_stop_rec = tk.Button(frame_buttons, text="⛔ Stop Recording", command=stop_recording,
                         font=("Arial", 10), width=16)
btn_stop_rec.grid(row=0, column=1, padx=6, pady=6)

btn_upload = tk.Button(frame_buttons, text="📁 Upload Audio File", command=on_upload_button,
                       font=("Arial", 12), width=22)
btn_upload.grid(row=1, column=0, padx=6, pady=6)

btn_play = tk.Button(frame_buttons, text="▶ Play Last Recording", command=on_play_last,
                     font=("Arial", 12), width=22)
btn_play.grid(row=1, column=1, padx=6, pady=6)

btn_stop_play = tk.Button(frame_buttons, text="⏹ Stop Playback", command=stop_audio,
                          font=("Arial", 10), width=16)
btn_stop_play.grid(row=2, column=1, padx=6, pady=6)

# Status & result labels
label_status = tk.Label(root, text="Waiting for input...", font=("Arial", 11))
label_status.pack(pady=10)

label_result = tk.Label(root, text="Prediction: ---", font=("Arial", 14), fg="blue")
label_result.pack(pady=10)

# History & exit
frame_bottom = tk.Frame(root)
frame_bottom.pack(pady=10)

btn_history = tk.Button(frame_bottom, text="🕘 Show History", command=show_history, width=18)
btn_history.grid(row=0, column=0, padx=8)

btn_quit = tk.Button(frame_bottom, text="❌ Exit", command=root.destroy, width=12)
btn_quit.grid(row=0, column=1, padx=8)

# Footer note for missing files
if model is None or scaler is None or encoder is None:
    footer_text = "Warning: model/scaler/encoder not loaded — prediction won't work until loaded."
else:
    footer_text = "Model loaded. Ready to predict."
footer = tk.Label(root, text=footer_text, font=("Arial", 9), fg="gray")
footer.pack(side="bottom", pady=8)

# Run the app
root.mainloop()
